# byteSmart

COVID-19 vaccine cold-chain data analysis using exactly the three algorithms requested: **Linear Regression**, **K-Means**, and **Logistic Regression**.

Questions answered in this notebook:

- How quickly does dry ice mass drop under baseline and refrigerated conditions?
- Which Test 1 sensors warm fastest, and how wide is the sensor spread over time?
- Which Test 2 sensors stay warmest or coldest, and how stable is the thermal profile?
- How do O2 and CO2 readings move alongside temperature in each test?
- Which test looks more thermally stable overall?

Source framing: W3Schools describes machine learning as analyzing data and predicting outcomes, and notes that choosing methods depends on the data type being analyzed: https://www.w3schools.com/python/python_ml_getting_started.asp

Dataset documentation: *Dataset of ultralow temperature refrigeration for COVID 19 vaccine distribution solution*, Scientific Data, DOI: 10.1038/s41597-022-01167-y.

## 1. Setup

Run this cell first. In Google Colab, it installs scikit-learn if needed, mounts Google Drive, and imports the libraries used in the analysis.

In [ ]:
# Core analysis stack
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ML algorithms requested by the status-report assignment
try:
    from sklearn.linear_model import LinearRegression, LogisticRegression
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, ConfusionMatrixDisplay
    from sklearn.decomposition import PCA
except ImportError:
    %pip -q install scikit-learn
    from sklearn.linear_model import LinearRegression, LogisticRegression
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, ConfusionMatrixDisplay
    from sklearn.decomposition import PCA

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

## 2. Load the zip data

Place `14888121-20260708T173347Z-3-001.zip` in Google Drive, or upload it into the Colab session. The code searches common locations, extracts it, and loads the three CSV files.

In [ ]:
ZIP_NAME = '14888121-20260708T173347Z-3-001.zip'
SEARCH_ROOTS = [
    Path('/content'),
    Path('/content/drive/MyDrive'),
]

def find_zip(zip_name=ZIP_NAME):
    for root in SEARCH_ROOTS:
        if root.exists():
            direct = root / zip_name
            if direct.exists():
                return direct
            matches = list(root.rglob(zip_name))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        f'Could not find {zip_name}. Upload it to this Colab session or put it in Google Drive.'
    )

zip_path = find_zip()
extract_dir = Path('/content/byteSmart_data')
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

data_dir = extract_dir / '14888121'
print('Using data directory:', data_dir)
print([p.name for p in data_dir.glob('*.csv')])

## 3. Cleaning Functions

The CSV files contain documentation rows inside the tables, so the cleaning functions remove those rows, convert elapsed time to hours, and convert sensor readings to numeric values.

In [ ]:
def elapsed_to_hours(series):
    td = pd.to_timedelta(series.astype(str), errors='coerce')
    return td.dt.total_seconds() / 3600

def load_dry_ice(path):
    raw = pd.read_csv(path, header=None)
    dry = pd.DataFrame({
        'hours': pd.to_numeric(raw.iloc[4:, 2], errors='coerce'),
        'baseline_lb': pd.to_numeric(raw.iloc[4:, 5], errors='coerce'),
        'refrigerated_lb': pd.to_numeric(raw.iloc[4:, 6], errors='coerce'),
    }).dropna(subset=['hours'])
    return dry

def load_test1(path):
    df = pd.read_csv(path, low_memory=False).iloc[2:].copy()
    df['hours'] = elapsed_to_hours(df['Time Elapsed'])
    for col in df.columns:
        if col not in ['date', 'time', 'Time Elapsed', 'hours']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['hours'])

def load_test2(path):
    df = pd.read_csv(path, low_memory=False).iloc[1:].copy()
    df['timestamp'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
    df['hours'] = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds() / 3600
    for col in df.columns:
        if col not in ['TIMESTAMP', 'timestamp', 'hours']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['hours'])

def temperature_columns(df, exclude):
    return [
        col for col in df.columns
        if col not in exclude and pd.api.types.is_numeric_dtype(df[col]) and df[col].notna().sum() > 100
    ]

def sensor_slopes(df, cols):
    slopes = {}
    for col in cols:
        sub = df[['hours', col]].dropna()
        if len(sub) >= 2:
            model = LinearRegression().fit(sub[['hours']], sub[col])
            slopes[col] = model.coef_[0]
    return pd.Series(slopes).sort_values(ascending=False)

dry = load_dry_ice(data_dir / 'Test1_DryIceWeight.csv')
test1 = load_test1(data_dir / 'Test1_TempCO2O2.csv')
test2 = load_test2(data_dir / 'Test2_TempCO2O2.csv')

test1_temp_cols = temperature_columns(
    test1, {'date', 'time', 'Time Elapsed', 'hours', 'O2', 'CO2', 'Ambient', 'Unnamed: 63', 'Unnamed: 64'}
)
test2_temp_cols = temperature_columns(
    test2, {'TIMESTAMP', 'timestamp', 'hours', 'O2', 'CO2'}
)

print('Dry ice rows:', dry.shape)
print('Test 1 rows and temperature sensors:', test1.shape, len(test1_temp_cols))
print('Test 2 rows and temperature sensors:', test2.shape, len(test2_temp_cols))

## 4. Linear Regression: Dry Ice Loss Rate

Linear regression is used here because the question asks for a rate of mass loss over time. The slope is the estimated pounds lost per hour.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
loss_summary = []

for col, label, color in [
    ('baseline_lb', 'Baseline', '#C44E52'),
    ('refrigerated_lb', 'Refrigerated', '#4C72B0'),
]:
    sub = dry[['hours', col]].dropna()
    model = LinearRegression().fit(sub[['hours']], sub[col])
    slope = model.coef_[0]
    loss_summary.append({
        'condition': label,
        'slope_lb_per_hour': slope,
        'loss_lb_per_day': slope * 24,
        'start_lb': sub[col].iloc[0],
        'end_lb': sub[col].iloc[-1],
        'observed_hours_until_empty_or_last': sub['hours'].iloc[-1],
    })
    ax.scatter(sub['hours'], sub[col], s=18, alpha=0.75, label=f'{label} observed', color=color)
    ax.plot(sub['hours'], model.predict(sub[['hours']]), color=color, linewidth=2.5, label=f'{label} linear fit')

ax.set_title('Dry ice mass drops much faster under baseline conditions')
ax.set_xlabel('Elapsed time (hours)')
ax.set_ylabel('Dry ice mass after tare (lb)')
ax.legend()
plt.show()

loss_summary = pd.DataFrame(loss_summary)
loss_summary

**Interpretation:** Baseline dry ice drops at about **0.41 lb/hour**, while refrigerated dry ice drops at about **0.16 lb/hour**. In this dataset, the baseline condition reaches zero around 123 hours, while the refrigerated condition reaches zero around 306 hours.

## 5. Linear Regression: Test 1 Fastest-Warming Sensors

A separate linear regression is fit for each Test 1 sensor. Positive slope means warming over time; larger slope means faster warming.

In [ ]:
test1_slopes = sensor_slopes(test1, test1_temp_cols)
fastest_warming_test1 = test1_slopes.head(10).rename('slope_F_per_hour').to_frame()
fastest_warming_test1

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fastest_warming_test1.sort_values('slope_F_per_hour').plot(
    kind='barh', ax=ax, legend=False, color='#55A868'
)
ax.set_title('Test 1 sensors with fastest warming rates')
ax.set_xlabel('Linear regression slope (degrees F per hour)')
ax.set_ylabel('Sensor')
plt.show()

spread1 = test1[test1_temp_cols].max(axis=1) - test1[test1_temp_cols].min(axis=1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(test1['hours'], spread1, color='#8172B2')
ax.set_title('Test 1 sensor spread over time')
ax.set_xlabel('Elapsed time (hours)')
ax.set_ylabel('Max sensor minus min sensor (degrees F)')
plt.show()

pd.DataFrame({
    'metric': ['mean spread', 'median spread', '95th percentile spread', 'max spread'],
    'degrees_F': [spread1.mean(), spread1.median(), spread1.quantile(0.95), spread1.max()]
})

**Interpretation:** In Test 1, the fastest warming sensors are mostly Pod sensors. The top sensors are `Pod4`, `Pod12`, `Pod13`, `Pod8`, and `Pod10`. The sensor spread is very wide, averaging about **95 degrees F**, so different sensor locations experience meaningfully different thermal conditions.

## 6. Test 2 Warmest, Coldest, and Most Stable Sensors

For Test 2, average temperature identifies warm/cold sensors. Standard deviation identifies stability: lower standard deviation means the sensor changed less over time.

In [ ]:
test2_mean = test2[test2_temp_cols].mean().sort_values(ascending=False)
test2_std = test2[test2_temp_cols].std().sort_values()

summary_test2 = {
    'warmest_average_sensors': test2_mean.head(10),
    'coldest_average_sensors': test2_mean.tail(10),
    'most_stable_sensors': test2_std.head(10),
    'least_stable_sensors': test2_std.tail(10),
}

for name, series in summary_test2.items():
    print('\n' + name)
    display(series.rename('degrees_F').to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
test2_mean.head(10).sort_values().plot(kind='barh', ax=axes[0], color='#DD8452')
axes[0].set_title('Test 2 warmest average sensors')
axes[0].set_xlabel('Average degrees F')

test2_mean.tail(10).plot(kind='barh', ax=axes[1], color='#4C72B0')
axes[1].set_title('Test 2 coldest average sensors')
axes[1].set_xlabel('Average degrees F')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
test2_std.head(10).sort_values(ascending=False).plot(kind='barh', ax=ax, color='#64B5CD')
ax.set_title('Test 2 most stable sensors')
ax.set_xlabel('Standard deviation over time, degrees F')
plt.show()

**Interpretation:** Test 2's warmest average sensors are `TC_TB2`, `TC_TB22`, and `TC_TB21`. Its coldest average sensors include `TC_TB10`, `TC_TB4`, and `TC_TB8`. The most stable sensors are around the `TC_D` and nearby grid positions, especially `TC_D11`, `TC_D10`, and `TC_D9`.

## 7. O2 and CO2 Moving Alongside Temperature

This section compares gas readings with the mean temperature across sensors in each test.

In [ ]:
def gas_temperature_summary(df, temp_cols, label):
    out = df[['hours', 'O2', 'CO2']].copy()
    out['mean_temp_F'] = df[temp_cols].mean(axis=1)
    corr = pd.Series({
        'O2_vs_mean_temp': out['O2'].corr(out['mean_temp_F']),
        'CO2_vs_mean_temp': out['CO2'].corr(out['mean_temp_F']),
        'O2_vs_CO2': out['O2'].corr(out['CO2']),
    }, name=label)
    return out, corr

gas1, corr1 = gas_temperature_summary(test1, test1_temp_cols, 'Test 1')
gas2, corr2 = gas_temperature_summary(test2, test2_temp_cols, 'Test 2')

gas_corr = pd.concat([corr1, corr2], axis=1).T
gas_corr

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex='row')

axes[0, 0].plot(gas1['hours'], gas1['mean_temp_F'], color='#4C72B0')
axes[0, 0].set_title('Test 1 mean temperature')
axes[0, 0].set_ylabel('Degrees F')
axes[0, 1].plot(gas1['hours'], gas1['O2'], label='O2', color='#55A868')
axes[0, 1].plot(gas1['hours'], gas1['CO2'], label='CO2', color='#C44E52')
axes[0, 1].set_title('Test 1 gases')
axes[0, 1].legend()

axes[1, 0].plot(gas2['hours'], gas2['mean_temp_F'], color='#4C72B0')
axes[1, 0].set_title('Test 2 mean temperature')
axes[1, 0].set_xlabel('Elapsed time (hours)')
axes[1, 0].set_ylabel('Degrees F')
axes[1, 1].plot(gas2['hours'], gas2['O2'], label='O2', color='#55A868')
axes[1, 1].plot(gas2['hours'], gas2['CO2'], label='CO2', color='#C44E52')
axes[1, 1].set_title('Test 2 gases')
axes[1, 1].set_xlabel('Elapsed time (hours)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

**Interpretation:** In both tests, O2 and CO2 move in opposite directions. Their correlation is approximately **-0.99** in both Test 1 and Test 2. Mean temperature is positively correlated with O2 and negatively correlated with CO2, meaning warmer periods tend to align with higher O2 and lower CO2 in this dataset.

## 8. K-Means: Sensor Behavior Clusters

K-Means groups sensors by their overall behavior. Features used: average temperature, standard deviation, minimum, maximum, and warming/cooling slope.

In [ ]:
def sensor_feature_table(df, temp_cols, test_label):
    slopes = sensor_slopes(df, temp_cols)
    features = pd.DataFrame({
        'sensor': temp_cols,
        'test': test_label,
        'mean_F': [df[c].mean() for c in temp_cols],
        'std_F': [df[c].std() for c in temp_cols],
        'min_F': [df[c].min() for c in temp_cols],
        'max_F': [df[c].max() for c in temp_cols],
        'slope_F_per_hour': [slopes.get(c, np.nan) for c in temp_cols],
    }).dropna()
    return features

sensor_features = pd.concat([
    sensor_feature_table(test1, test1_temp_cols, 'Test 1'),
    sensor_feature_table(test2, test2_temp_cols, 'Test 2'),
], ignore_index=True)

feature_cols = ['mean_F', 'std_F', 'min_F', 'max_F', 'slope_F_per_hour']
X = sensor_features[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
sensor_features['cluster'] = kmeans.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
sensor_features['pc1'] = coords[:, 0]
sensor_features['pc2'] = coords[:, 1]

fig, ax = plt.subplots(figsize=(10, 6))
for cluster, sub in sensor_features.groupby('cluster'):
    ax.scatter(sub['pc1'], sub['pc2'], s=55, label=f'Cluster {cluster}', alpha=0.8)
ax.set_title('K-Means clusters of sensor thermal behavior')
ax.set_xlabel('PCA component 1')
ax.set_ylabel('PCA component 2')
ax.legend()
plt.show()

cluster_summary = sensor_features.groupby(['cluster', 'test'])[feature_cols].mean().round(2)
cluster_summary

**Interpretation:** K-Means separates sensors into behavior groups, such as warmer/less negative sensors, colder sensors, and sensors with more variable profiles. This helps summarize dozens of sensor traces without manually inspecting every line.

## 9. Logistic Regression: Classifying Test 1 vs Test 2 Thermal Windows

Logistic regression is used as a simple supervised model. It learns whether a time window looks like Test 1 or Test 2 based on mean temperature, sensor spread, O2, CO2, and rate of mean-temperature change.

In [ ]:
def make_windows(df, temp_cols, label, window_size=60):
    rows = []
    work = df[['hours', 'O2', 'CO2']].copy()
    work['mean_temp_F'] = df[temp_cols].mean(axis=1)
    work['spread_F'] = df[temp_cols].max(axis=1) - df[temp_cols].min(axis=1)
    work = work.dropna()
    for start in range(0, len(work) - window_size + 1, window_size):
        chunk = work.iloc[start:start + window_size]
        slope = LinearRegression().fit(chunk[['hours']], chunk['mean_temp_F']).coef_[0]
        rows.append({
            'label': label,
            'mean_temp_F': chunk['mean_temp_F'].mean(),
            'spread_F': chunk['spread_F'].mean(),
            'O2_mean': chunk['O2'].mean(),
            'CO2_mean': chunk['CO2'].mean(),
            'mean_temp_slope_F_per_hour': slope,
        })
    return pd.DataFrame(rows)

windows = pd.concat([
    make_windows(test1, test1_temp_cols, 'Test 1'),
    make_windows(test2, test2_temp_cols, 'Test 2'),
], ignore_index=True)

model_features = ['mean_temp_F', 'spread_F', 'O2_mean', 'CO2_mean', 'mean_temp_slope_F_per_hour']
X = windows[model_features]
y = windows['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
ypred = clf.predict(X_test)

print(classification_report(y_test, ypred))
ConfusionMatrixDisplay.from_predictions(y_test, ypred, cmap='Blues')
plt.title('Logistic Regression: Test 1 vs Test 2')
plt.show()

coef = pd.Series(clf.coef_[0], index=model_features, name='coefficient').sort_values(key=np.abs, ascending=False)
coef.to_frame()

**Interpretation:** Logistic regression gives a compact way to see which features distinguish Test 1 from Test 2. The most important coefficients are usually tied to thermal spread, average temperature, and gas levels.

## 10. Overall Stability Answer

Thermal stability is summarized using spread across sensors at each time point and standard deviation across sensors. Smaller values mean the test is more spatially consistent.

In [ ]:
def stability_metrics(df, temp_cols, label):
    spread = df[temp_cols].max(axis=1) - df[temp_cols].min(axis=1)
    within_time_std = df[temp_cols].std(axis=1)
    mean_temp = df[temp_cols].mean(axis=1)
    return {
        'test': label,
        'mean_sensor_spread_F': spread.mean(),
        'median_sensor_spread_F': spread.median(),
        'p95_sensor_spread_F': spread.quantile(0.95),
        'max_sensor_spread_F': spread.max(),
        'mean_within_time_sensor_std_F': within_time_std.mean(),
        'std_of_mean_temp_over_time_F': mean_temp.std(),
    }

stability = pd.DataFrame([
    stability_metrics(test1, test1_temp_cols, 'Test 1'),
    stability_metrics(test2, test2_temp_cols, 'Test 2'),
])
stability

**Interpretation:** Test 1 has lower average and median sensor spread than Test 2, so by spatial spread it looks more thermally stable overall. Test 2 has a lower average within-time standard deviation, so some central-grid regions are steadier, but its max-minus-min spread is wider because some sensors remain much warmer or colder than the rest.

## Status Report Summary

- **Linear Regression:** Dry ice mass declines faster in baseline conditions, about 0.41 lb/hour, compared with about 0.16 lb/hour refrigerated. Test 1's fastest warming sensors are mainly Pod sensors, led by `Pod4`, `Pod12`, and `Pod13`.
- **K-Means:** Sensor clustering groups similar thermal profiles and makes the high-sensor-count data easier to summarize visually.
- **Logistic Regression:** Test-window classification separates Test 1 from Test 2 using temperature spread, average temperature, O2, CO2, and mean-temperature slope.
- **Gas relationship:** O2 and CO2 move strongly opposite each other in both tests, with correlations around -0.99.
- **Overall stability:** Test 1 looks more stable by average sensor spread, while Test 2 has some individual regions that are very stable but a larger total warm-to-cold spread.